# Thực hành ở nhà Transformers

Hoàn thiện hàm huấn luyện cho mạng Transformer và tiến hành huấn luyện mô hình

### Cài đặt giải thuật tối ưu và huấn luyện mô hình

In [ ]:
!python -m spacy download fr_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 28.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 99.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import spacy
from collections import Counter

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [22]:
def create_masks(src, trg_input, src_pad, trg_pad):
  # src_key_padding_mask: (Batch, Src_Len), True means mask (padding token)
  src_key_padding_mask = (src == src_pad).to(device)

  # trg_key_padding_mask for trg_input: (Batch, Trg_Input_Len), True means mask (padding token)
  # This mask is for the decoder's self-attention where `trg_input` is the query/key/value.
  trg_input_key_padding_mask = (trg_input == trg_pad).to(device)

  # tgt_attn_mask: (Trg_Input_Len, Trg_Input_Len), causal mask, True means mask (upper triangle)
  seq_len_trg_input = trg_input.size(1)
  # This creates an upper triangular matrix with `True` for positions that should be masked.
  # diagonal=1 makes sure the diagonal itself is not masked.
  tgt_attn_mask = torch.triu(torch.ones(seq_len_trg_input, seq_len_trg_input), diagonal=1).bool().to(device)

  return src_key_padding_mask, trg_input_key_padding_mask, tgt_attn_mask

In [20]:
class TransformerModel(nn.Module):
  def __init__(self, src_vocab, trg_vocab, d_model=512, nhead=8, num_layers = 6, dropout=0.1):
    super().__init__()
    self.src_embed = nn.Embedding(src_vocab, d_model)
    self.trg_embed = nn.Embedding(trg_vocab, d_model)

    self.transformer = nn.Transformer(d_model=d_model,
                                      nhead=nhead,
                                      num_encoder_layers=num_layers,
                                      num_decoder_layers=num_layers,
                                      dropout=dropout, batch_first=True)
    self.fc_out = nn.Linear(d_model, trg_vocab)

  # Changed parameters to reflect standard Transformer masks
  def forward(self, src, trg, src_key_padding_mask, tgt_key_padding_mask, tgt_attn_mask):
    src_emb = self.src_embed(src)
    trg_emb = self.trg_embed(trg)
    out = self.transformer(src_emb, trg_emb,
                           src_key_padding_mask=src_key_padding_mask,
                           tgt_key_padding_mask=tgt_key_padding_mask,
                           memory_key_padding_mask=src_key_padding_mask, # memory_key_padding_mask uses src_key_padding_mask
                           tgt_mask=tgt_attn_mask
                          )
    return self.fc_out(out)

In [ ]:
class TranslationDataset(Dataset):
  def __init__(self, src_path, trg_path, src_tokenizer, trg_tokenizer, src_vocab, trg_vocab, max_len=80):
    self.src_lines = open(src_path, encoding='utf-8').read().split('\n')
    self.trg_lines = open(trg_path, encoding='utf-8').read().split('\n')
    self.data = []
    for s, t in zip(self.src_lines, self.trg_lines):
      if not s.strip() or not t.strip():
        continue
      src_tokens = ['<sos>'] + [tok.text.lower() for tok in src_tokenizer(s)] + ['<eos>']
      trg_tokens = ['<sos>'] + [tok.text.lower() for tok in trg_tokenizer(t)] + ['<eos>']
      src_ids = [src_vocab.get(tok, src_vocab['<unk>']) for tok in src_tokens][:max_len]
      trg_ids = [trg_vocab.get(tok, trg_vocab['<unk>']) for tok in trg_tokens][:max_len]
      self.data.append((src_ids, trg_ids))

  def __len__(self):
    return len(self.data)

  def __getitem__(self, idx):
    return self.data[idx]

In [15]:
def collate_fn(batch, src_pad, trg_pad):
  src_seqs, trg_seqs = zip(*batch)
  src_max_len = max(len(s) for s in src_seqs)
  trg_max_len = max(len(t) for t in trg_seqs)

  src_padded = torch.tensor([s + [src_pad]*(src_max_len - len(s)) for s in src_seqs])
  trg_padded = torch.tensor([t + [trg_pad]*(trg_max_len - len(t)) for t in trg_seqs])

  return src_padded, trg_padded

In [ ]:
def build_vocab(sentences, tokenizer, min_freq=2):
  counter = Counter()
  for line in sentences:
    tokens = [tok.text.lower() for tok in tokenizer(line)]
    counter.update(tokens)
  vocab = {'<pad>':0, '<sos>':1, '<eos>':2, '<unk>':3}
  for word, freq in counter.items():
    if freq >= min_freq:
      vocab[word] = len(vocab)
  return vocab

In [ ]:
import os

os.makedirs("data", exist_ok=True)

english_text = """I am a student.
You are a teacher.
He is eating.
We are learning.
They are playing football.
She likes apples.
It is raining.
I love programming.
You speak French.
We study together.
"""

french_text = """Je suis un étudiant.
Tu es un professeur.
Il mange.
Nous apprenons.
Ils jouent au football.
Elle aime les pommes.
Il pleut.
J'aime la programmation.
Tu parles français.
Nous étudions ensemble.
"""

with open("data/english.txt", "w") as f:
    f.write(english_text)

with open("data/french.txt", "w") as f:
    f.write(french_text)

print("✅ Data files created successfully!")


✅ Data files created successfully!


In [21]:
import time

""" BAI TAP VE NHA """

def train_model(model, dataloader, optimizer, src_pad, trg_pad, epochs=2, printevery=50):
  print("Training model...")
  model.train()
  start = time.time()

  for epoch in range(epochs):
    total_loss = 0
    for i, (src, trg) in enumerate(dataloader):
      src, trg = src.to(device), trg.to(device)
      trg_input = trg[:, :-1]

      # Call create_masks to get the new mask types
      src_key_padding_mask, trg_input_key_padding_mask, tgt_attn_mask = create_masks(src, trg_input, src_pad, trg_pad)

      # Pass the masks to the model with correct keyword arguments
      preds = model(src, trg_input,
                    src_key_padding_mask=src_key_padding_mask,
                    tgt_key_padding_mask=trg_input_key_padding_mask,
                    tgt_attn_mask=tgt_attn_mask)

      ys = trg[:, 1:].contiguous().view(-1)
      optimizer.zero_grad()
      loss = F.cross_entropy(preds.view(-1, preds.size(-1)), ys, ignore_index=trg_pad)
      loss.backward()
      optimizer.step()
      total_loss += loss.item()
      if (i+1) % printevery == 0:
        avg_loss = total_loss / printevery
        print(f"Epoch {epoch+1} [{i+1}/{len(dataloader)}] Loss = {avg_loss:.3f}")
        total_loss = 0
    print(f"{int((time.time()-start)//60)}m: Epoch {epoch+1} complete")

class Opt:
  pass

def main():
  opt = Opt()
  opt.src_data = "data/english.txt"
  opt.trg_data = "data/french.txt"
  opt.src_lang = "en_core_web_sm"
  opt.trg_lang = 'fr_core_news_sm'
  opt.epochs = 2
  opt.d_model=512
  opt.n_layers=6
  opt.heads=8
  opt.dropout=0.1
  opt.batchsize=1500
  opt.printevery=100
  opt.lr=1e-4
  opt.max_strlen=80
  opt.checkpoint = 0
  opt.no_cuda = False
  opt.load_weights = None

  opt.device = 0
  if opt.device == 0:
    assert torch.cuda.is_available()

  # Tokenizers
  opt.src_tokenizer = spacy.load(opt.src_lang)
  opt.trg_tokenizer = spacy.load(opt.trg_lang)

  # Build vocab
  src_lines = open(opt.src_data, encoding='utf8').read().split('\n')
  trg_lines = open(opt.trg_data, encoding='utf8').read().split('\n')
  src_vocab = build_vocab(src_lines, opt.src_tokenizer)
  trg_vocab = build_vocab(trg_lines, opt.trg_tokenizer)

  opt.src_pad = src_vocab['<pad>']
  opt.trg_pad = trg_vocab['<pad>']

  # Dataset and DataLoader
  dataset = TranslationDataset(opt.src_data, opt.trg_data,
                               opt.src_tokenizer, opt.trg_tokenizer,
                               src_vocab, trg_vocab, max_len=opt.max_strlen)

  dataloader = DataLoader(dataset, batch_size=opt.batchsize, shuffle=True,
                         collate_fn=lambda b: collate_fn(b, opt.src_pad, opt.trg_pad))

  model = TransformerModel(len(src_vocab), len(trg_vocab), d_model=opt.d_model,
                           nhead=opt.heads, num_layers=opt.n_layers, dropout=opt.dropout).to(device)
  optimizer = torch.optim.Adam(model.parameters(), lr=opt.lr, betas=(0.9, 0.98), eps=1e-9)

  train_model(model, dataloader, optimizer, opt.src_pad, opt.trg_pad, epochs=opt.epochs, printevery=opt.printevery)
    # for asking about further training use while true loop, and return
if __name__ == "__main__":
    main()

Training model...
0m: Epoch 1 complete
0m: Epoch 2 complete
